# Standalone Evaluation — BiomedCLIP + MLP with MedGemma Augmentation

This notebook skips all training and performs evaluation only.

It supports:
- **Phase 1 embedding evaluation** using cached fused embeddings (`fused_emb_val.npz` / `fused_emb_test.npz`)
- **Optional true Phase 2 evaluation** from raw images + cached pseudo reports, if you build the corresponding raw-image loader.

It computes:
- Macro F1
- Micro F1
- Hamming Accuracy
- Exact Match Accuracy
- Macro Sensitivity
- Macro Specificity
- Macro Youden-J
- ROC-AUC (macro / micro)
- Fabrication Error Rate (FER)
- FER for abnormal studies
- Omission Rate
- Overall contingency counts (TP / FP / FN / TN)
- Per-label metrics and contingency counts

It also saves four CSV files:
- summary
- per-label
- overall contingency
- per-label contingency

In [12]:
# ============================================================
# 0) Configuration
# ============================================================
from __future__ import annotations
from pathlib import Path

# -------- Model IDs --------
BIOCLIP_MODEL_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"

# -------- Data paths --------
# Choose validation or test split here.
SPLIT_NAME = "val"   # options: "val", "test"
CACHE_DIR  = Path("/data/liangz2/openi/biomedclip_mimic_13label_cache")

VAL_JSONL  = Path("/data/liangz2/openi/faiss_val_mimic_biomedclip/metadata.jsonl")
TEST_JSONL = Path("/data/liangz2/openi/faiss_test_mimic_biomedclip/metadata.jsonl")  # adjust if needed

PSEUDO_REPORT_CACHE = CACHE_DIR / "pseudo_reports.jsonl"
FUSED_EMB_VAL_CACHE = CACHE_DIR / "fused_emb_val.npz"
FUSED_EMB_TEST_CACHE = CACHE_DIR / "fused_emb_test.npz"   # adjust if needed

MODEL_SAVE_PATH_P1 = CACHE_DIR / "mlp_classifier_phase1_best.pt"
MODEL_SAVE_PATH_P2 = CACHE_DIR / "mlp_classifier_phase2_best.pt"

# -------- Evaluation mode --------
# phase1_embeddings: evaluate using cached 1024-d fused embeddings
# phase2_images: true end-to-end evaluation from raw images + cached pseudo reports
EVAL_MODE = "phase2_images"

# -------- Output --------
OUTPUT_DIR = CACHE_DIR
OUTPUT_PREFIX = OUTPUT_DIR / f"biomedclip_medgemma_{SPLIT_NAME}_{EVAL_MODE}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------- Misc --------
BATCH_SIZE = 64
THRESHOLD  = 0.5
NUM_WORKERS = 0

LABELS_13 = [
    "atelectasis",
    "cardiomegaly",
    "consolidation",
    "edema",
    "enlarged cardiomediastinum",
    "fracture",
    "lung lesion",
    "lung opacity",
    "pleural effusion",
    "pleural other",
    "pneumonia",
    "pneumothorax",
    "support devices",
]
NUM_LABELS = len(LABELS_13)

if SPLIT_NAME == "val":
    TARGET_JSONL = VAL_JSONL
    TARGET_FUSED_CACHE = FUSED_EMB_VAL_CACHE
elif SPLIT_NAME == "test":
    TARGET_JSONL = TEST_JSONL
    TARGET_FUSED_CACHE = FUSED_EMB_TEST_CACHE
else:
    raise ValueError(f"Unsupported SPLIT_NAME: {SPLIT_NAME}")

print("Config ready.")
print("  SPLIT_NAME       :", SPLIT_NAME)
print("  EVAL_MODE        :", EVAL_MODE)
print("  TARGET_JSONL     :", TARGET_JSONL)
print("  TARGET_FUSED     :", TARGET_FUSED_CACHE)
print("  MODEL_SAVE_PATH_P1:", MODEL_SAVE_PATH_P1)
print("  MODEL_SAVE_PATH_P2:", MODEL_SAVE_PATH_P2)
print("  OUTPUT_PREFIX    :", OUTPUT_PREFIX)

Config ready.
  SPLIT_NAME       : val
  EVAL_MODE        : phase2_images
  TARGET_JSONL     : /data/liangz2/openi/faiss_val_mimic_biomedclip/metadata.jsonl
  TARGET_FUSED     : /data/liangz2/openi/biomedclip_mimic_13label_cache/fused_emb_val.npz
  MODEL_SAVE_PATH_P1: /data/liangz2/openi/biomedclip_mimic_13label_cache/mlp_classifier_phase1_best.pt
  MODEL_SAVE_PATH_P2: /data/liangz2/openi/biomedclip_mimic_13label_cache/mlp_classifier_phase2_best.pt
  OUTPUT_PREFIX    : /data/liangz2/openi/biomedclip_mimic_13label_cache/biomedclip_medgemma_val_phase2_images


In [13]:
# ============================================================
# 1) Imports
# ============================================================
import csv
import json
import re
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import open_clip

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    hamming_loss,
    multilabel_confusion_matrix,
)

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x  # noqa: E731

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Imports ready.")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Device:", device)

✅ Imports ready.
PyTorch: 2.9.1+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Device: cuda


In [14]:
# ============================================================
# 2) Helpers: dataset loading and pseudo-report cache
# ============================================================
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_gt_labels_binary(row: Dict[str, Any]) -> np.ndarray:
    lab_raw = row.get("labels", [])
    if isinstance(lab_raw, str):
        try:
            lab_raw = json.loads(lab_raw)
        except Exception:
            lab_raw = [x.strip() for x in lab_raw.split(",") if x.strip()]
    lab_set = {str(l).strip().lower() for l in lab_raw}
    return np.array([1 if lab in lab_set else 0 for lab in LABELS_13], dtype=np.float32)

def load_pseudo_report_cache(cache_path: Path) -> Dict[str, str]:
    cache: Dict[str, str] = {}
    if not cache_path.exists():
        raise FileNotFoundError(f"Pseudo report cache not found: {cache_path}")
    with open(cache_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                row = json.loads(line)
                cache[row["image_path"]] = row.get("pseudo_report", "")
    return cache

records = load_jsonl(TARGET_JSONL)
print(f"Loaded {len(records)} records from {TARGET_JSONL}")

pseudo_report_cache = load_pseudo_report_cache(PSEUDO_REPORT_CACHE)
print(f"Loaded {len(pseudo_report_cache)} pseudo reports from {PSEUDO_REPORT_CACHE}")

Loaded 634 records from /data/liangz2/openi/faiss_val_mimic_biomedclip/metadata.jsonl
Loaded 11539 pseudo reports from /data/liangz2/openi/biomedclip_mimic_13label_cache/pseudo_reports.jsonl


In [15]:
# ============================================================
# 3) Load BiomedCLIP
# ============================================================
_bioclip_device = device

biomedclip_model, biomedclip_preprocess = open_clip.create_model_from_pretrained(BIOCLIP_MODEL_ID)
biomedclip_tokenizer = open_clip.get_tokenizer(BIOCLIP_MODEL_ID)
biomedclip_model = biomedclip_model.to(_bioclip_device).eval()

# Resolve dimensions
with torch.no_grad():
    _dummy_img  = torch.zeros(1, 3, 224, 224).to(_bioclip_device)
    _dummy_tok  = biomedclip_tokenizer(["normal chest"]).to(_bioclip_device)
    _img_feat   = biomedclip_model.encode_image(_dummy_img)
    _txt_feat   = biomedclip_model.encode_text(_dummy_tok)

EMBED_DIM = int(_img_feat.shape[-1])
FUSED_DIM = EMBED_DIM * 2

print(f"✅ BiomedCLIP loaded on {_bioclip_device}")
print(f"   EMBED_DIM = {EMBED_DIM}")
print(f"   FUSED_DIM = {FUSED_DIM}")

✅ BiomedCLIP loaded on cuda
   EMBED_DIM = 512
   FUSED_DIM = 1024


In [16]:
# ============================================================
# 4) Model definitions
# ============================================================
class MLPClassifier(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = 13, dropout: float = 0.3):
        super().__init__()
        self.norm_in = nn.LayerNorm(input_dim)
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout * 0.67),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(self.norm_in(x))


class BiomedCLIPWithMLP(nn.Module):
    def __init__(self, biomedclip: nn.Module, mlp: MLPClassifier):
        super().__init__()
        self.biomedclip = biomedclip
        self.mlp = mlp

    def forward(self, images: torch.Tensor, txt_tokens: torch.Tensor) -> torch.Tensor:
        img_feat = self.biomedclip.encode_image(images)
        txt_feat = self.biomedclip.encode_text(txt_tokens)

        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True).clamp_min(1e-12)
        txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True).clamp_min(1e-12)

        fused = torch.cat([img_feat, txt_feat], dim=-1)
        logits = self.mlp(fused)
        return logits


mlp = MLPClassifier(input_dim=FUSED_DIM, num_classes=NUM_LABELS, dropout=0.3).to(device)
full_model = BiomedCLIPWithMLP(biomedclip_model, mlp).to(device)

print("✅ Model objects instantiated.")

✅ Model objects instantiated.


In [17]:
# ============================================================
# 5) Datasets / dataloaders
# ============================================================
class FusedEmbeddingDataset(Dataset):
    def __init__(self, embeddings: np.ndarray, labels: np.ndarray):
        self.embeddings = embeddings.astype(np.float32)
        self.labels = labels.astype(np.float32)

    def __len__(self) -> int:
        return len(self.embeddings)

    def __getitem__(self, idx: int):
        return torch.from_numpy(self.embeddings[idx]), torch.from_numpy(self.labels[idx])


class ImageTextDataset(Dataset):
    def __init__(
        self,
        records: List[Dict[str, Any]],
        pseudo_cache: Dict[str, str],
        preprocess,
        tokenizer,
    ):
        self.records = records
        self.pseudo_cache = pseudo_cache
        self.preprocess = preprocess
        self.tokenizer = tokenizer

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        row = self.records[idx]
        ip = row["image_path"]
        pseudo = self.pseudo_cache.get(ip, "")
        gt = extract_gt_labels_binary(row)

        img = Image.open(ip).convert("RGB")
        img_tensor = self.preprocess(img)
        txt_tokens = self.tokenizer([pseudo])[0]

        return img_tensor, txt_tokens, torch.from_numpy(gt)


# Build phase-1 embedding loader
if not TARGET_FUSED_CACHE.exists():
    raise FileNotFoundError(
        f"Fused embedding cache not found: {TARGET_FUSED_CACHE}. "
        f"Create it in the training notebook first."
    )

_data = np.load(TARGET_FUSED_CACHE)
target_embeddings = _data["embeddings"]
target_labels = _data["labels"]
target_emb_ds = FusedEmbeddingDataset(target_embeddings, target_labels)
target_emb_loader = DataLoader(
    target_emb_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print("✅ Fused embedding loader ready:", target_embeddings.shape, target_labels.shape)

# Optional phase-2 raw image loader
target_imgtext_loader = None
if EVAL_MODE == "phase2_images":
    target_imgtext_ds = ImageTextDataset(
        records=records,
        pseudo_cache=pseudo_report_cache,
        preprocess=biomedclip_preprocess,
        tokenizer=biomedclip_tokenizer,
    )
    target_imgtext_loader = DataLoader(
        target_imgtext_ds,
        batch_size=min(BATCH_SIZE, 16),
        shuffle=False,
        num_workers=NUM_WORKERS,
    )
    print("✅ Raw image+text loader ready:", len(target_imgtext_ds))

✅ Fused embedding loader ready: (634, 1024) (634, 13)
✅ Raw image+text loader ready: 634


In [18]:
# ============================================================
# 6) Metrics
# ============================================================
def compute_comprehensive_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_score: Optional[np.ndarray],
    threshold: float = THRESHOLD,
) -> Dict[str, Any]:
    N = y_true.shape[0]
    n_labels = y_true.shape[1]
    total_label_slots = int(N * n_labels)

    macro_f1    = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    micro_f1    = float(f1_score(y_true, y_pred, average="micro", zero_division=0))
    per_f1      = f1_score(y_true, y_pred, average=None, zero_division=0)
    hamming_acc = float(1.0 - hamming_loss(y_true, y_pred))
    exact_match = float((y_true == y_pred).all(axis=1).mean())

    total_fp_labels = int(((y_pred == 1) & (y_true == 0)).sum())
    total_fn_labels = int(((y_pred == 0) & (y_true == 1)).sum())
    total_tp_labels = int(((y_pred == 1) & (y_true == 1)).sum())
    total_tn_labels = int(((y_pred == 0) & (y_true == 0)).sum())

    total_predicted_positive_labels = int((y_pred == 1).sum())
    total_gt_positive_labels = int((y_true == 1).sum())

    fabrication_error_rate = (
        float(total_fp_labels / total_predicted_positive_labels)
        if total_predicted_positive_labels > 0 else 0.0
    )
    omission_rate = (
        float(total_fn_labels / total_gt_positive_labels)
        if total_gt_positive_labels > 0 else 0.0
    )

    abnormal_mask = (y_true.sum(axis=1) > 0)
    num_abnormal_cases = int(abnormal_mask.sum())
    abnormal_cases_with_pred_positive = (
        int((y_pred[abnormal_mask].sum(axis=1) > 0).sum())
        if num_abnormal_cases > 0 else 0
    )
    abnormal_pred = y_pred[abnormal_mask]
    abnormal_true = y_true[abnormal_mask]
    abnormal_fp_labels = (
        int(((abnormal_pred == 1) & (abnormal_true == 0)).sum())
        if num_abnormal_cases > 0 else 0
    )
    abnormal_predicted_positive_labels = (
        int((abnormal_pred == 1).sum())
        if num_abnormal_cases > 0 else 0
    )
    fer_abnormal = (
        float(abnormal_fp_labels / abnormal_predicted_positive_labels)
        if abnormal_predicted_positive_labels > 0 else 0.0
    )

    mcm = multilabel_confusion_matrix(y_true, y_pred)
    per_label: Dict[str, Any] = {}
    sens_l, spec_l, yj_l = [], [], []

    for i, lab in enumerate(LABELS_13):
        tn, fp, fn, tp = mcm[i].ravel()
        sens = tp / (tp + fn + 1e-9) if (tp + fn) > 0 else float("nan")
        spec = tn / (tn + fp + 1e-9) if (tn + fp) > 0 else float("nan")
        youden = (sens + spec - 1.0) if not (np.isnan(sens) or np.isnan(spec)) else float("nan")

        per_label[lab] = {
            "TP": int(tp),
            "FP": int(fp),
            "TN": int(tn),
            "FN": int(fn),
            "f1": float(per_f1[i]),
            "sensitivity": float(sens),
            "specificity": float(spec),
            "youden_j": float(youden),
            "roc_auc": float("nan"),
        }

        if not np.isnan(sens):
            sens_l.append(sens)
        if not np.isnan(spec):
            spec_l.append(spec)
        if not np.isnan(youden):
            yj_l.append(youden)

    roc_macro = float("nan")
    roc_micro = float("nan")
    if y_score is not None:
        try:
            roc_macro = float(roc_auc_score(y_true, y_score, average="macro"))
            roc_micro = float(roc_auc_score(y_true, y_score, average="micro"))
            per_auc = roc_auc_score(y_true, y_score, average=None)
            for i, lab in enumerate(LABELS_13):
                per_label[lab]["roc_auc"] = float(per_auc[i])
        except Exception as e:
            print(f"⚠️ ROC-AUC: {e}")

    return {
        "N": N,
        "threshold": threshold,
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "hamming_accuracy": hamming_acc,
        "exact_match_accuracy": exact_match,
        "macro_sensitivity": float(np.mean(sens_l)) if sens_l else float("nan"),
        "macro_specificity": float(np.mean(spec_l)) if spec_l else float("nan"),
        "macro_youden_j": float(np.mean(yj_l)) if yj_l else float("nan"),
        "roc_auc_macro": roc_macro,
        "roc_auc_micro": roc_micro,
        "fabrication_error_rate": fabrication_error_rate,
        "fer_abnormal": fer_abnormal,
        "omission_rate": omission_rate,
        "total_tp_labels": total_tp_labels,
        "total_fp_labels": total_fp_labels,
        "total_fn_labels": total_fn_labels,
        "total_tn_labels": total_tn_labels,
        "total_predicted_positive_labels": total_predicted_positive_labels,
        "total_gt_positive_labels": total_gt_positive_labels,
        "total_label_slots": total_label_slots,
        "num_abnormal_cases": num_abnormal_cases,
        "abnormal_cases_with_pred_positive": abnormal_cases_with_pred_positive,
        "per_label": per_label,
    }


def print_metrics_table(m: Dict[str, Any]) -> None:
    print("\n" + "=" * 96)
    print("  EVALUATION METRICS SUMMARY")
    print("=" * 96)
    print(f"  N samples                         : {m['N']}")
    print(f"  Threshold                         : {m['threshold']}")
    print(f"  Macro F1                          : {m['macro_f1']:.4f}")
    print(f"  Micro F1                          : {m['micro_f1']:.4f}")
    print(f"  Hamming Accuracy                  : {m['hamming_accuracy']:.4f}")
    print(f"  Exact Match Accuracy              : {m['exact_match_accuracy']:.4f}")
    print(f"  Macro Sensitivity                 : {m['macro_sensitivity']:.4f}")
    print(f"  Macro Specificity                 : {m['macro_specificity']:.4f}")
    print(f"  Macro Youden-J                    : {m['macro_youden_j']:.4f}")
    print(f"  ROC-AUC (macro)                   : {m['roc_auc_macro']}")
    print(f"  ROC-AUC (micro)                   : {m['roc_auc_micro']}")
    print(f"  Fabrication Error Rate (FER)      : {m['fabrication_error_rate']:.4f}")
    print(f"  FER for abnormal studies          : {m['fer_abnormal']:.4f}")
    print(f"  Omission Rate                     : {m['omission_rate']:.4f}")
    print()
    print(f"  Total TP labels                   : {m['total_tp_labels']}")
    print(f"  Total FP labels                   : {m['total_fp_labels']}")
    print(f"  Total FN labels                   : {m['total_fn_labels']}")
    print(f"  Total TN labels                   : {m['total_tn_labels']}")
    print(f"  Total predicted positive labels   : {m['total_predicted_positive_labels']}")
    print(f"  Total GT positive labels          : {m['total_gt_positive_labels']}")
    print(f"  # abnormal cases                  : {m['num_abnormal_cases']}")
    print(f"  # abnormal cases w/ pred positive : {m['abnormal_cases_with_pred_positive']}")

    H = f"\n  {'Label':<32} {'F1':>6} {'Sens':>6} {'Spec':>6} {'Youden':>7} {'AUC':>6}"
    print(H)
    print("  " + "-" * (len(H) - 3))
    for lab, lm in m["per_label"].items():
        print(
            f"  {lab:<32} "
            f"{lm['f1']:>6.3f} "
            f"{lm['sensitivity']:>6.3f} "
            f"{lm['specificity']:>6.3f} "
            f"{lm['youden_j']:>7.3f} "
            f"{lm.get('roc_auc', float('nan')):>6}"
        )
    print("=" * 96 + "\n")


print("✅ Metrics helpers ready.")

✅ Metrics helpers ready.


In [19]:
# ============================================================
# 7) Prediction helpers
# ============================================================
def _collect_predictions_mlp(
    mlp_model: MLPClassifier,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    mlp_model.eval()
    all_logits, all_labels = [], []
    with torch.inference_mode():
        for emb, labels in tqdm(loader, desc="eval embeddings"):
            emb = emb.to(device, dtype=torch.float32)
            logits = mlp_model(emb)
            all_logits.append(logits.cpu())
            all_labels.append(labels.cpu())
    all_logits = torch.cat(all_logits, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy().astype(int)
    all_probs = torch.sigmoid(torch.from_numpy(all_logits)).numpy()
    all_preds = (all_probs >= THRESHOLD).astype(int)
    return all_labels, all_preds, all_probs


def _collect_predictions_e2e(
    model: BiomedCLIPWithMLP,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    all_logits, all_labels = [], []
    with torch.inference_mode():
        for imgs, txt_tokens, labels in tqdm(loader, desc="eval e2e"):
            imgs = imgs.to(device, dtype=torch.float32)
            txt_tokens = txt_tokens.to(device)
            logits = model(imgs, txt_tokens)
            all_logits.append(logits.cpu())
            all_labels.append(labels.cpu())
    all_logits = torch.cat(all_logits, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy().astype(int)
    all_probs = torch.sigmoid(torch.from_numpy(all_logits)).numpy()
    all_preds = (all_probs >= THRESHOLD).astype(int)
    return all_labels, all_preds, all_probs

In [20]:
# ============================================================
# 8) Load saved checkpoint and run evaluation
# ============================================================
if EVAL_MODE == "phase2_images":
    if target_imgtext_loader is None:
        raise RuntimeError("target_imgtext_loader is not available for phase2_images evaluation.")
    if not MODEL_SAVE_PATH_P2.exists():
        raise FileNotFoundError(f"Phase 2 checkpoint not found: {MODEL_SAVE_PATH_P2}")

    ckpt = torch.load(MODEL_SAVE_PATH_P2, map_location=device)

    if "model_state" in ckpt:
        full_model.load_state_dict(ckpt["model_state"], strict=False)
    else:
        if "biomedclip_state" in ckpt:
            biomedclip_model.load_state_dict(ckpt["biomedclip_state"], strict=False)
        if "encoder_state" in ckpt:
            biomedclip_model.load_state_dict(ckpt["encoder_state"], strict=False)
        if "mlp_state" in ckpt:
            mlp.load_state_dict(ckpt["mlp_state"], strict=False)

    print(f"✅ Loaded Phase 2 checkpoint: {MODEL_SAVE_PATH_P2}")
    print(f"   epoch={ckpt.get('epoch', '?')}  macro_f1={ckpt.get('val_macro_f1', float('nan')):.4f}")
    y_true, y_pred, y_score = _collect_predictions_e2e(full_model, target_imgtext_loader, device)

elif EVAL_MODE == "phase1_embeddings":
    if not MODEL_SAVE_PATH_P1.exists():
        raise FileNotFoundError(f"Phase 1 checkpoint not found: {MODEL_SAVE_PATH_P1}")

    ckpt = torch.load(MODEL_SAVE_PATH_P1, map_location=device)
    mlp.load_state_dict(ckpt["mlp_state"], strict=False)

    print(f"✅ Loaded Phase 1 checkpoint: {MODEL_SAVE_PATH_P1}")
    print(f"   epoch={ckpt.get('epoch', '?')}  macro_f1={ckpt.get('val_macro_f1', float('nan')):.4f}")
    y_true, y_pred, y_score = _collect_predictions_mlp(mlp, target_emb_loader, device)

else:
    raise ValueError(f"Unknown EVAL_MODE: {EVAL_MODE}")

print("Shapes:")
print("  y_true :", y_true.shape)
print("  y_pred :", y_pred.shape)
print("  y_score:", y_score.shape)

✅ Loaded Phase 2 checkpoint: /data/liangz2/openi/biomedclip_mimic_13label_cache/mlp_classifier_phase2_best.pt
   epoch=14  macro_f1=0.2951


eval e2e:   0%|          | 0/40 [00:00<?, ?it/s]

Shapes:
  y_true : (634, 13)
  y_pred : (634, 13)
  y_score: (634, 13)


In [21]:
# ============================================================
# 9) Compute metrics
# ============================================================
eval_metrics = compute_comprehensive_metrics(
    y_true=y_true,
    y_pred=y_pred,
    y_score=y_score,
    threshold=THRESHOLD,
)

print_metrics_table(eval_metrics)


  EVALUATION METRICS SUMMARY
  N samples                         : 634
  Threshold                         : 0.5
  Macro F1                          : 0.2951
  Micro F1                          : 0.3876
  Hamming Accuracy                  : 0.8175
  Exact Match Accuracy              : 0.2571
  Macro Sensitivity                 : 0.4564
  Macro Specificity                 : 0.8389
  Macro Youden-J                    : 0.2952
  ROC-AUC (macro)                   : 0.7285912243812068
  ROC-AUC (micro)                   : 0.7829281561690109
  Fabrication Error Rate (FER)      : 0.7043
  FER for abnormal studies          : 0.6471
  Omission Rate                     : 0.4374

  Total TP labels                   : 476
  Total FP labels                   : 1134
  Total FN labels                   : 370
  Total TN labels                   : 6262
  Total predicted positive labels   : 1610
  Total GT positive labels          : 846
  # abnormal cases                  : 416
  # abnormal cases w/ pr

In [22]:
# ============================================================
# 10) Save metrics to CSV
# ============================================================
def save_single_model_metrics_to_csv(
    metrics: Dict[str, Any],
    output_prefix: str | Path,
    model_name: str = "BiomedCLIP + MLP + MedGemma augment",
    n_labels: int = 13,
):
    output_prefix = Path(output_prefix)
    output_prefix.parent.mkdir(parents=True, exist_ok=True)

    # 1) summary
    summary_row = {
        "model": model_name,
        "N": metrics.get("N"),
        "threshold": metrics.get("threshold"),
        "macro_f1": metrics.get("macro_f1"),
        "micro_f1": metrics.get("micro_f1"),
        "hamming_accuracy": metrics.get("hamming_accuracy"),
        "exact_match_accuracy": metrics.get("exact_match_accuracy"),
        "macro_sensitivity": metrics.get("macro_sensitivity"),
        "macro_specificity": metrics.get("macro_specificity"),
        "macro_youden_j": metrics.get("macro_youden_j"),
        "roc_auc_macro": metrics.get("roc_auc_macro"),
        "roc_auc_micro": metrics.get("roc_auc_micro"),
        "fabrication_error_rate": metrics.get("fabrication_error_rate"),
        "fer_abnormal": metrics.get("fer_abnormal"),
        "omission_rate": metrics.get("omission_rate"),
        "total_tp_labels": metrics.get("total_tp_labels"),
        "total_fp_labels": metrics.get("total_fp_labels"),
        "total_fn_labels": metrics.get("total_fn_labels"),
        "total_tn_labels": metrics.get("total_tn_labels"),
        "total_predicted_positive_labels": metrics.get("total_predicted_positive_labels"),
        "total_gt_positive_labels": metrics.get("total_gt_positive_labels"),
        "num_abnormal_cases": metrics.get("num_abnormal_cases"),
        "abnormal_cases_with_pred_positive": metrics.get("abnormal_cases_with_pred_positive"),
    }
    df_summary = pd.DataFrame([summary_row])
    summary_path = output_prefix.with_name(output_prefix.name + "_summary.csv")
    df_summary.to_csv(summary_path, index=False)

    # 2) per-label
    per_label_rows = []
    for label_name, label_metrics in metrics.get("per_label", {}).items():
        per_label_rows.append({
            "model": model_name,
            "label": label_name,
            "TP": label_metrics.get("TP"),
            "FP": label_metrics.get("FP"),
            "TN": label_metrics.get("TN"),
            "FN": label_metrics.get("FN"),
            "f1": label_metrics.get("f1"),
            "sensitivity": label_metrics.get("sensitivity"),
            "specificity": label_metrics.get("specificity"),
            "youden_j": label_metrics.get("youden_j"),
            "roc_auc": label_metrics.get("roc_auc"),
        })
    df_per_label = pd.DataFrame(per_label_rows)
    per_label_path = output_prefix.with_name(output_prefix.name + "_per_label.csv")
    df_per_label.to_csv(per_label_path, index=False)

    # 3) overall contingency
    n_samples = int(metrics["N"])
    total_labels = n_samples * n_labels
    fp = int(metrics["total_fp_labels"])
    fn = int(metrics["total_fn_labels"])
    gt_pos = int(metrics["total_gt_positive_labels"])
    pred_pos = int(metrics["total_predicted_positive_labels"])
    tp = gt_pos - fn
    tn = total_labels - tp - fp - fn

    df_contingency = pd.DataFrame([{
        "model": model_name,
        "N_samples": n_samples,
        "Total_Labels": total_labels,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Predicted_Positive": pred_pos,
        "GroundTruth_Positive": gt_pos,
        "Num_Abnormal_Cases": metrics.get("num_abnormal_cases"),
        "Abnormal_Cases_With_Pred_Positive": metrics.get("abnormal_cases_with_pred_positive"),
    }])
    contingency_path = output_prefix.with_name(output_prefix.name + "_contingency.csv")
    df_contingency.to_csv(contingency_path, index=False)

    # 4) per-label contingency
    per_label_cont_rows = []
    for label_name, label_metrics in metrics.get("per_label", {}).items():
        per_label_cont_rows.append({
            "model": model_name,
            "label": label_name,
            "TP": label_metrics.get("TP"),
            "FP": label_metrics.get("FP"),
            "FN": label_metrics.get("FN"),
            "TN": label_metrics.get("TN"),
        })
    df_per_label_cont = pd.DataFrame(per_label_cont_rows)
    per_label_cont_path = output_prefix.with_name(output_prefix.name + "_per_label_contingency.csv")
    df_per_label_cont.to_csv(per_label_cont_path, index=False)

    print(f"✅ Saved summary CSV               -> {summary_path}")
    print(f"✅ Saved per-label CSV             -> {per_label_path}")
    print(f"✅ Saved overall contingency CSV   -> {contingency_path}")
    print(f"✅ Saved per-label contingency CSV -> {per_label_cont_path}")

    return df_summary, df_per_label, df_contingency, df_per_label_cont


import pandas as pd

df_summary, df_per_label, df_contingency, df_per_label_contingency = save_single_model_metrics_to_csv(
    metrics=eval_metrics,
    output_prefix=OUTPUT_PREFIX,
    model_name=f"BiomedCLIP+MLP+MedGemma ({SPLIT_NAME}, {EVAL_MODE})",
)

print("\nPreview: summary")
display(df_summary.head())

print("\nPreview: per-label")
display(df_per_label.head())

✅ Saved summary CSV               -> /data/liangz2/openi/biomedclip_mimic_13label_cache/biomedclip_medgemma_val_phase2_images_summary.csv
✅ Saved per-label CSV             -> /data/liangz2/openi/biomedclip_mimic_13label_cache/biomedclip_medgemma_val_phase2_images_per_label.csv
✅ Saved overall contingency CSV   -> /data/liangz2/openi/biomedclip_mimic_13label_cache/biomedclip_medgemma_val_phase2_images_contingency.csv
✅ Saved per-label contingency CSV -> /data/liangz2/openi/biomedclip_mimic_13label_cache/biomedclip_medgemma_val_phase2_images_per_label_contingency.csv

Preview: summary


,model,N,threshold,macro_f1,micro_f1,hamming_accuracy,exact_match_accuracy,macro_sensitivity,macro_specificity,macro_youden_j,...,fer_abnormal,omission_rate,total_tp_labels,total_fp_labels,total_fn_labels,total_tn_labels,total_predicted_positive_labels,total_gt_positive_labels,num_abnormal_cases,abnormal_cases_with_pred_positive
0,"BiomedCLIP+MLP+MedGemma (val, phase2_images)",634,0.5,0.295148,0.387622,0.81752,0.257098,0.456351,0.838888,0.295239,...,0.647146,0.437352,476,1134,370,6262,1610,846,416,307



Preview: per-label


,model,label,TP,FP,TN,FN,f1,sensitivity,specificity,youden_j,roc_auc
0,"BiomedCLIP+MLP+MedGemma (val, phase2_images)",atelectasis,59,142,400,33,0.402730,0.641304,0.738007,0.379312,0.726797
1,"BiomedCLIP+MLP+MedGemma (val, phase2_images)",cardiomegaly,74,146,379,35,0.449848,0.678899,0.721905,0.400804,0.731289
2,"BiomedCLIP+MLP+MedGemma (val, phase2_images)",consolidation,12,44,561,17,0.282353,0.413793,0.927273,0.341066,0.794528
3,"BiomedCLIP+MLP+MedGemma (val, phase2_images)",edema,31,66,508,29,0.394904,0.516667,0.885017,0.401684,0.821196
4,"BiomedCLIP+MLP+MedGemma (val, phase2_images)",enlarged cardiomediastinum,4,49,570,11,0.117647,0.266667,0.920840,0.187507,0.640388
